In [16]:
%%writefile reduction_replay.cu
#include <cuda_runtime.h>

#include <algorithm>
#include <cmath>
#include <iomanip>
#include <iostream>
#include <string>
#include <vector>

__global__ void reduce_sum_kernel(
    const float* input,
    float* output,
    int n,
    bool faulty
) {
    extern __shared__ float shared[];

    int tid = threadIdx.x;
    float local_sum = 0.0f;

    for (int i = tid; i < n; i += blockDim.x) {
        if (!faulty || i < n - 1) {
            local_sum += input[i];
        }
    }

    shared[tid] = local_sum;
    __syncthreads();

    for (int stride = blockDim.x / 2; stride > 0; stride /= 2) {
        if (tid < stride) {
            shared[tid] += shared[tid + stride];
        }
        __syncthreads();
    }

    if (tid == 0) {
        output[0] = shared[0];
    }
}

bool check_cuda(cudaError_t status, const char* operation) {
    if (status == cudaSuccess) {
        return true;
    }

    std::cerr << operation << ": "
              << cudaGetErrorString(status) << '\n';
    return false;
}

int main() {
    std::string implementation;
    int n = 0;
    int block_size = 0;

    if (!(std::cin >> implementation >> n >> block_size)) {
        std::cerr << "Invalid replay header\n";
        return 2;
    }

    if (implementation != "faulty_tail") {
        std::cerr << "Unsupported implementation: "
                  << implementation << '\n';
        return 2;
    }

    if (n <= 0 || block_size <= 0) {
        std::cerr << "Invalid shape or block size\n";
        return 2;
    }

    std::vector<float> input(n);

    for (float& value : input) {
        if (!(std::cin >> value)) {
            std::cerr << "Not enough input values\n";
            return 2;
        }
    }

    cudaDeviceProp device{};
    if (!check_cuda(
            cudaGetDeviceProperties(&device, 0),
            "cudaGetDeviceProperties")) {
        return 1;
    }

    std::cerr << "GPU: " << device.name << '\n';
    std::cerr << "Compute capability: "
              << device.major << "." << device.minor << '\n';

    float* device_input = nullptr;
    float* device_output = nullptr;

    if (!check_cuda(
            cudaMalloc(
                &device_input,
                input.size() * sizeof(float)),
            "cudaMalloc(input)")) {
        return 1;
    }

    if (!check_cuda(
            cudaMalloc(&device_output, sizeof(float)),
            "cudaMalloc(output)")) {
        cudaFree(device_input);
        return 1;
    }

    if (!check_cuda(
            cudaMemcpy(
                device_input,
                input.data(),
                input.size() * sizeof(float),
                cudaMemcpyHostToDevice),
            "cudaMemcpy(input)")) {
        cudaFree(device_input);
        cudaFree(device_output);
        return 1;
    }

    reduce_sum_kernel<<<
        1,
        block_size,
        block_size * sizeof(float)
    >>>(
        device_input,
        device_output,
        n,
        true
    );

    if (!check_cuda(
            cudaDeviceSynchronize(),
            "cudaDeviceSynchronize")) {
        cudaFree(device_input);
        cudaFree(device_output);
        return 1;
    }

    float actual = 0.0f;

    if (!check_cuda(
            cudaMemcpy(
                &actual,
                device_output,
                sizeof(float),
                cudaMemcpyDeviceToHost),
            "cudaMemcpy(output)")) {
        cudaFree(device_input);
        cudaFree(device_output);
        return 1;
    }

    cudaFree(device_input);
    cudaFree(device_output);

    double expected = 0.0;

    for (float value : input) {
        expected += static_cast<double>(value);
    }

    double tolerance =
        1e-5 * std::max(1.0, std::abs(expected));

    bool mismatch =
        std::abs(static_cast<double>(actual) - expected)
        > tolerance;

    std::cout << std::setprecision(9)
              << "{"
              << "\"n\":" << n << ","
              << "\"block_threads\":" << block_size << ","
              << "\"expected\":" << expected << ","
              << "\"actual\":" << actual << ","
              << "\"mismatch\":"
              << (mismatch ? "true" : "false")
              << "}\n";

    return 0;
}

Writing reduction_replay.cu


In [17]:
!nvcc -std=c++20 -O0 -lineinfo -arch=sm_75 \
    reduction_replay.cu \
    -o reduction_replay

In [18]:
import json
import math
import subprocess
from pathlib import Path

failure_path = Path("failure.json")
assert failure_path.exists(), "failure.json is missing"

failure = json.loads(failure_path.read_text())

implementation = failure["implementation"]
values = failure["input"]["values"]
n = failure["input"]["shape"][0]
block_threads = failure["launch"]["block_threads"]

recorded_expected = failure["observed"]["expected"]
recorded_actual = failure["observed"]["actual"]

assert implementation == "faulty_tail"
assert n == len(values)

payload = (
    f"{implementation}\n"
    f"{n}\n"
    f"{block_threads}\n"
    + " ".join(str(value) for value in values)
    + "\n"
)

process = subprocess.run(
    ["./reduction_replay"],
    input=payload,
    text=True,
    capture_output=True,
)

print(process.stderr)

if process.returncode != 0:
    raise RuntimeError(
        f"Replay infrastructure failed:\n{process.stderr}"
    )

result = json.loads(process.stdout)

if not math.isclose(
    result["expected"],
    recorded_expected,
    rel_tol=0.0,
    abs_tol=1e-6,
):
    classification = "different_failure"
elif not result["mismatch"]:
    classification = "resolved"
elif math.isclose(
    result["actual"],
    recorded_actual,
    rel_tol=0.0,
    abs_tol=1e-6,
):
    classification = "same_failure"
else:
    classification = "different_failure"

replay_result = {
    "artifact": str(failure_path),
    "classification": classification,
    "recorded_actual": recorded_actual,
    "replayed_actual": result["actual"],
    "replayed_expected": result["expected"],
    "block_threads": block_threads,
    "n": n,
}

Path("replay_result.json").write_text(
    json.dumps(replay_result, indent=2)
)

print(json.dumps(replay_result, indent=2))

assert classification == "same_failure"
print("REPLAY PASS: same failure reproduced")

GPU: Tesla T4
Compute capability: 7.5

{
  "artifact": "failure.json",
  "classification": "same_failure",
  "recorded_actual": 30,
  "replayed_actual": 30,
  "replayed_expected": 31,
  "block_threads": 32,
  "n": 31
}
REPLAY PASS: same failure reproduced
